# GMM clustering notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.R -> dimensionality_reduction.R -> gmm_clustering.ipynb`

In [10]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [11]:
# Imports
from pathlib import Path

import numpy as np
import pandas as pd
from gapstat import gapstat_score
from scipy.stats import chi2_contingency

from utils.bootstrap_utils import BootstrapGridSearch, nan_if_single_cluster
from utils.clustering_utils import predict_clustering_labels, predict_clustering_proba
from utils.gmm_utils import GaussianMixture
from utils.metrics_utils import (
    adjusted_rand_score,
    association,
    calinski_harabasz_score,
    davies_bouldin_score,
    dunn_index,
    normalized_mutual_info_score,
    silhouette_score,
    variation_of_information,
)
from utils.misc_utils import contingency_matrix
from utils.saving_utils import DatasetBundle, save_object

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [12]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
assert data_discov_full.meta is not None
assert data_discov_full.na is not None
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
assert data_valid_full.meta is not None
assert data_valid_full.na is not None

data_discov_mfa = pd.read_parquet(Path("data/dim_red/data_discov_mfa.parquet"))
data_valid_mfa = pd.read_parquet(Path("data/dim_red/data_valid_mfa.parquet"))

# Set outputs folder
output_folder = Path("outputs/gmm_clustering")
output_folder.mkdir(exist_ok=True)

# Set sample weights
sample_weight_discov = data_discov_full.data["wichtn"].to_numpy()
sample_weight_valid = data_valid_full.data["wichtn"].to_numpy()

Loaded dataset from data/preprocessed/data_discov_full.
Loaded dataset from data/preprocessed/data_valid_full.


In [ ]:
# Create RNG for reproducibility
rng0 = np.random.default_rng(352532)
rng1, rng2, rng3 = rng0.spawn(3)

reg_covar = 1e-3  # Standard wäre 1e-6

gmm_discov = GaussianMixture(random_state=np.random.RandomState(rng0.bit_generator), covariance_type="full",  reg_covar=reg_covar)
gmm_valid = GaussianMixture(random_state=np.random.RandomState(rng1.bit_generator), covariance_type="full", reg_covar=reg_covar)

n_boot = 50
param_grid = {"n_components": range(2, 51)}
inbag_metrics = {
    "bic": lambda e, X, labels, sample_weight: e.bic(X, sample_weight=sample_weight),
    "icl": lambda e, X, labels, sample_weight: e.icl(X, sample_weight=sample_weight),
    "aic": lambda e, X, labels, sample_weight: e.aic(X, sample_weight=sample_weight),
}
oob_metrics = {
    "loglike": lambda e, X, labels, sample_weight: e.score(X, sample_weight=sample_weight),
    "dbi": nan_if_single_cluster(
        lambda e, X, labels, sample_weight: davies_bouldin_score(X, labels, sample_weight=sample_weight)
    ),
    "chi": nan_if_single_cluster(
        lambda e, X, labels, sample_weight: calinski_harabasz_score(X, labels, sample_weight=sample_weight)
    ),
    "si": nan_if_single_cluster(
        lambda e, X, labels, sample_weight: silhouette_score(X, labels, sample_weight=sample_weight)
    ),
    "di": nan_if_single_cluster(lambda e, X, labels, sample_weight: dunn_index(X, labels)),
}
agreement_metrics = {
    "ari": lambda x, y, sample_weight: adjusted_rand_score(x, y, sample_weight=sample_weight),
    "nmi": lambda x, y, sample_weight: normalized_mutual_info_score(x, y, sample_weight=sample_weight),
    "vi": lambda x, y, sample_weight: variation_of_information(x, y, sample_weight=sample_weight),
    "assoc": lambda x, y, sample_weight: association(
        contingency_matrix(x, y, sample_weight=sample_weight), correction=False
    ),
    "chi2": lambda x, y, sample_weight: (
        chi2_contingency(contingency_matrix(x, y, sample_weight=sample_weight), correction=False).statistic
    ),
}

# Run bootstrap grid-search loop on discovery data
gmm_boot_discov = BootstrapGridSearch(
    gmm_discov,
    param_grid,
    n_boot=n_boot,
    inbag_metrics=inbag_metrics,
    oob_metrics=oob_metrics,
    agreement_metrics=agreement_metrics,
    selection_metric="loglike",
    tie_break=None,
    use_nonnegative_amendment=True,
    rng=rng2,
)
gmm_boot_discov = gmm_boot_discov.fit(data_discov_mfa, sample_weight=sample_weight_discov, verbose=True)

# Fit validation data with only the best parameters from the discovery search
gmm_boot_valid = BootstrapGridSearch(
    gmm_valid,
    {k: [v] for k, v in gmm_boot_discov.best_params_.items()},
    n_boot=n_boot,
    inbag_metrics=inbag_metrics,
    oob_metrics=oob_metrics,
    agreement_metrics=agreement_metrics,
    selection_metric="loglike",
    tie_break=None,
    use_nonnegative_amendment=True,
    rng=rng3,
)
gmm_boot_valid = gmm_boot_valid.fit(data_valid_mfa, sample_weight=sample_weight_valid, verbose=False)

model_discov = gmm_boot_discov.best_estimator_
model_valid = gmm_boot_valid.best_estimator_

# Sort clusters independently by total survey weight, largest first
for model, data, sample_weight in (
    (model_discov, data_discov_mfa, sample_weight_discov),
    (model_valid, data_valid_mfa, sample_weight_valid),
):
    cluster_weight = np.bincount(model.predict(data), weights=sample_weight, minlength=model.n_components)
    order = np.argsort(-cluster_weight, kind="stable")
    model.weights_ = model.weights_[order]
    model.means_ = model.means_[order]
    if model.covariance_type != "tied":
        model.covariances_ = model.covariances_[order]
        model.precisions_ = model.precisions_[order]
        model.precisions_cholesky_ = model.precisions_cholesky_[order]

Loaded file from outputs/gmm_clustering/gmm_boot_discov.pkl.
Loaded file from outputs/gmm_clustering/model_discov.pkl.
Loaded file from outputs/gmm_clustering/model_valid.pkl.


/opt/venv/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but GaussianMixture was fitted without feature names
/opt/venv/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but GaussianMixture was fitted without feature names


In [14]:
# Apply models to predict labels
labels_discov = predict_clustering_labels(model_discov, data_discov_mfa)
labels_valid = predict_clustering_labels(model_valid, data_valid_mfa)

# Apply models to predict probabilities
proba_discov = predict_clustering_proba(model_discov, data_discov_mfa)
proba_valid = predict_clustering_proba(model_valid, data_valid_mfa)

/opt/venv/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but GaussianMixture was fitted without feature names
/opt/venv/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but GaussianMixture was fitted without feature names
/opt/venv/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but GaussianMixture was fitted without feature names
/opt/venv/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but GaussianMixture was fitted without feature names


### Compare clusterings
We are dealing with _inferential clustering_, because we are clustering individuals and not variables.\
Method-based validation: The same clustering method is applied to the validation data, and the results are compared between clusterings.\
Result-based validation: The trained cluster algorithm is applied to the validation data, and the results are compared between datasets.\
Cluster stability: The results from the trained and transferred clusterings from the method- and result-based validation steps are compared.

In [ ]:
# Assign metrics directions
metrics_directions = pd.Series(
    {
        "neg_mean_of_sq": "max",
        "loglike": "max",
        "bic": "min",
        "icl": "min",
        "aic": "min",
        "dbi": "min",
        "chi": "max",
        "si": "max",
        "di": "max",
        "ari": "max",
        "nmi": "max",
        "vi": "min",
        "assoc": "max",
        "chi2": "max",
    },
    name="direction",
).astype("category")
metrics_grid_search_discov = gmm_boot_discov.summaries_.merge(
    metrics_directions,
    left_on="metric",
    right_index=True,
    validate="many_to_one",
)
metrics_valid = gmm_boot_valid.summaries_.merge(
    metrics_directions,
    left_on="metric",
    right_index=True,
    validate="many_to_one",
)

# Compare clusterings
metrics_discov_valid_comparison = (
    metrics_grid_search_discov.merge(pd.DataFrame(gmm_boot_discov.best_params_, index=pd.Index([0])))
    .set_index(["metric", "scope", "n_components", "direction"])
    .join(
        metrics_valid.set_index(["metric", "scope", "n_components", "direction"]), lsuffix="_discov", rsuffix="_valid"
    )
    .reset_index()
    .drop(columns="n_components")
)
metrics_discov_valid_comparison

,metric,scope,direction,mean_discov,mcse_discov,std_discov,n_valid_discov,mean_valid,mcse_valid,std_valid,n_valid_valid
0,bic,inbag,min,-155316.038457,2241.717752,15851.338241,50,-138527.493019,2722.054175,19247.829661,50
1,icl,inbag,min,-155185.344845,2242.464965,15856.621835,50,-138371.701164,2724.129576,19262.504958,50
2,aic,inbag,min,-165935.442991,2241.717752,15851.338241,50,-149097.039705,2722.054175,19247.829661,50
3,loglike,oob,max,7.335133,0.224964,1.590733,50,5.604739,0.319909,2.262096,50
4,dbi,oob,min,2.066630,0.027507,0.194507,50,2.164571,0.031345,0.221639,50
5,chi,oob,max,179.746026,4.127524,29.186005,50,177.605768,3.307367,23.386613,50
6,si,oob,max,0.230745,0.003333,0.023569,50,0.204161,0.003653,0.025832,50
7,di,oob,max,0.008110,0.000557,0.003936,50,0.006349,0.000380,0.002686,50
8,ari,agreement,max,0.680847,0.020866,0.147543,50,0.720126,0.021500,0.152026,50
9,nmi,agreement,max,0.721824,0.012348,0.087315,50,0.679654,0.010212,0.072208,50


In [20]:
# Save grid-search results
metrics_grid_search_discov.to_parquet(output_folder / "metrics_grid_search_discov.parquet", index=None)

# Save models
save_object(gmm_boot_discov, output_folder / "gmm_boot_discov.pkl")
save_object(model_discov, output_folder / "model_discov.pkl")
save_object(model_valid, output_folder / "model_valid.pkl")

# Save predictions
proba_discov.assign(**{"label": labels_discov}).to_parquet(
    output_folder / "cluster_pred_discov.parquet", index=False
)
proba_valid.assign(**{"label": labels_valid}).to_parquet(
    output_folder / "cluster_pred_valid.parquet", index=False
)

# Save comparison
metrics_discov_valid_comparison.to_parquet(output_folder / "metrics_discov_valid_comparison.parquet", index=False)

Saved file to outputs/gmm_clustering/gmm_boot_discov.pkl.
Saved file to outputs/gmm_clustering/model_discov.pkl.
Saved file to outputs/gmm_clustering/model_valid.pkl.
